# Project: Text Generation (ChatBot)

- Date: October 30 2025-

- Data: The data used in that project come from a document called Machine Learning for absolute beginners made by Olivier Theobald.

- Description: In this third project,we will use RAG method. We will first use a pre-trained model, tuning it for our purpose and make it able to answer question on the data we will give to it. At the end, we will also build an interface for a better comunication with the model using React.js ( for the front-end) and Node.js (for the back-end).

In [1]:
import torch

In [2]:
#Just to make our computation faster in case we do not have GPU card
if torch.cuda.is_available():
  print("GPU is available.")
  device = torch.cuda.current_device()
else:
  print("Will work on CPU.")
  print(torch.get_num_threads())
  torch.set_num_threads(8)
  print("Cores used now:", torch.get_num_threads())


Will work on CPU.
4
Cores used now: 8


## Downloading and cleaning of the data

We can notice here that we already have the data we will use for this project, but we have to transform that pdf into a text for our machine learning tasks.

In [3]:
import pdfplumber
import os
import re
from sentence_transformers import SentenceTransformer
import numpy as np

/home/christian/ProjetsPerso/pytorch-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
#Download
pdf_name="/home/christian/ProjetsPerso/Artificial_Intelligence/DeepLearning/Text_Generation/data/Machine Learning For Absolute Beginners_ A Plain English Introduction (Second Edition) - Machine Learning For Absolute Beginners.pdf"
with pdfplumber.open(pdf_name) as pdf:
    completed_text=""
    for page in pdf.pages:
        completed_text+=page.extract_text()+"\n"

# We put it into a text file
text_name="/home/christian/ProjetsPerso/Artificial_Intelligence/DeepLearning/Text_Generation/data/text.txt"
if os.path.exists(text_name): os.remove(text_name)

with open(text_name, "w", encoding="utf-8") as text:
    text.write(completed_text)

Now we have our text we will clean it and change it into chunks and vetors.

In [5]:
texts=""
with open(text_name, 'r', encoding="utf-8") as f:
    texts=f.read()

# Cleaning part
##Every word, pontuation, symbole could be important for the understanding of the text
##So we decide to keep everything and to only delete successive space and emails.
print("Size before cleaning:", len(texts.split()))
texts=re.sub(pattern=r'\S+@\S+', repl=' ', string=texts)
texts=re.sub(pattern=r'\s+', repl=' ', string=texts) # notice that here the order is important :)


# Made it into chunks
texts=texts.split()
print("Size after cleaning:", len(texts))
CHUNK_SIZE=200

chunks=[" ".join(texts[i:i+CHUNK_SIZE]) for i in range(0,len(texts),CHUNK_SIZE)]

#Just a little display
print("===Display====")
for i in range(3):
    print(f"\tChunk {i} :", chunks[i])
print("Number of chunks:", len(chunks))


Size before cleaning: 24944
Size after cleaning: 24940
===Display====
	Chunk 0 : Machine Learning For Absolute Beginners Oliver Theobald Second Edition Copyright © 2017 by Oliver Theobald All rights reserved. No part of this publication may be reproduced, distributed, or transmitted in any form or by any means, including photocopying, recording, or other electronic or mechanical methods, without the prior written permission of the publisher, except in the case of brief quotations embodied in critical reviews and certain other non-commercial uses permitted by copyright law. Contents INTRODUCTION WHAT IS MACHINE LEARNING? ML CATEGORIES THE ML TOOLBOX DATA SCRUBBING SETTING UP YOUR DATA REGRESSION ANALYSIS CLUSTERING BIAS & VARIANCE ARTIFICIAL NEURAL NETWORKS DECISION TREES ENSEMBLE MODELING BUILDING A MODEL IN PYTHON MODEL OPTIMIZATION FURTHER RESOURCES DOWNLOADING DATASETS FINAL WORD INTRODUCTION Machines have come a long way since the Industrial Revolution. They continue to fill factor

In [6]:
t="JE       SUIS oh la vache anej nenf 123&'  "
print(t.split())
print(len(chunks)*CHUNK_SIZE)

['JE', 'SUIS', 'oh', 'la', 'vache', 'anej', 'nenf', "123&'"]
25000


## Creation of Embeddings and Storing 

In this part, we will create the embeddings using the modul sentence_tranformers and his function SentenceTransfomer.
We will load first a pre-trained model named "all-MiniLM-L6-v2".

In [7]:
model=SentenceTransformer("all-MiniLM-L6-v2")
print(model)

SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)


We can see here that this model can handle a sentence or a chunk or an input with a size max 256 (so we have no problem) and each chunk is transformed into an embedding of 384 dimensions.

In [8]:
embeddings=[model.encode(chunk) for chunk in chunks]
embeddings=np.array(embeddings, dtype="float32")
print(f"Embeddings created whith shape of {embeddings.shape}.")

Embeddings created whith shape of (125, 384).


#### Just to undersant how sentenceTransformer works

In [9]:
sentences=["I am very happy to see you.", "Happy is a very bad student.", "It was a wounderful encounter", "Who is Happy?"]
encoded_sentences=model.encode(sentences) 
encoded_sentences=np.array(encoded_sentences, dtype="float32")
print(encoded_sentences)
print(encoded_sentences.shape)
similarities = model.similarity(encoded_sentences, encoded_sentences)
print("===Similarities===\n", similarities)


[[-5.5149715e-02  3.3461127e-02  5.0490100e-02 ...  5.3471509e-02
  -8.9072973e-02 -4.9321651e-02]
 [ 5.8404334e-02  1.0884575e-01  6.3570091e-03 ...  3.4573019e-02
   1.4897736e-02  1.7929334e-02]
 [-6.0240380e-02  1.1208525e-01  1.9731170e-02 ... -1.9669583e-02
   3.2493877e-03 -1.7551931e-03]
 [-2.4942919e-05  6.1674524e-02  1.4970356e-03 ... -3.5813483e-04
   6.1735030e-02 -8.3218031e-03]]
(4, 384)
===Similarities===
 tensor([[1.0000, 0.3025, 0.0506, 0.4475],
        [0.3025, 1.0000, 0.1011, 0.5488],
        [0.0506, 0.1011, 1.0000, 0.0803],
        [0.4475, 0.5488, 0.0803, 1.0000]])


##
So now we have our embeddings, we want to save them for a later comparision with a query. And for this, we will use FAISS (Fast AI Similarity Search) which is a modul. This modul will also be usesul to find similarities for a query. 

In [10]:
# you have to use pip install faiss-cpu or faiss-gpu depending on if you have a gpu card or not.
import faiss

In [11]:
#Creation of the index
dim=embeddings.shape[1] 
index=faiss.IndexFlatIP(dim) #we pass dim as a parameter cause this function need to know the dimension of the vectors for allocating memory for storing
embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True) #normalization for the cosinus similarity
index.add(embeddings) #we add our data to that index

#storing
faiss.write_index(index, "./data/index.faiss")

# storing the chunks to retrieve the text later but not usefull now
# with open("chunks.json", "w", encoding="utf-8") as f:
#     json.dump(chunks, f, ensure_ascii=False, indent=2)

## Generation of the answer

So the first step to generate the answer is to build a prompt and send it to a GPT for the answer. The prompt is made by choosing the best chunks in term of similarities with the query. So we will find first what are the best similar chunks regarding the question (the query) and create a prompt for our generative model.

### Construction o fthe prompt

In [12]:
# we search first the chunks: but not usefull now
# index = faiss.read_index("index.faiss")
# with open("chunks.json", "r", encoding="utf-8") as f:
#     chunks = json.load(f)

#Get the query
query="What is Machine learning?" #we will change after to make it like a conversation
query_embedding=model.encode(query)
query_embedding=np.array([query_embedding], dtype='float32')
query_embedding=query_embedding/np.linalg.norm(query_embedding, axis=1, keepdims=True)

#Gather all the good chunks
D,I=index.search(query_embedding,k=3) #Here D correspond to a matrix of distance and I a matrix of index
context = "\n\n".join(chunks[i] for i in I[0])

prompt=f"""
Focusing on the following passages, build an anwser.

Text to focus on:{context}

Question:{query}
Answer(simple, precise with examples if possible):
"""


### Creation of the answers

We will use an openAI model for generating the text, this model is "gpt-3.5-turbo".

In [20]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import InferenceClient
from dotenv import load_dotenv #because I work in a virtual environement

#loading environment variable
load_dotenv()

client = InferenceClient(
    api_key=os.environ["HUGGINGFACEHUB_API_TOKEN"],
)

completion = client.chat.completions.create(
    model="meta-llama/Llama-3.2-1B-Instruct",
    messages=[
        {
            "role": "user",
            "content": prompt,
        }
    ],
)

print(completion.choices[0].message.content)

Machine learning is the ability of computers to learn and improve their performance from data and experience, without being explicitly programmed, by identifying patterns, making predictions, and adapting to new situations.
